# Fine-tune ViT5-base + LoRA — Vietnamese news summarization (dataset v2)

Companion notebook for **TICKET-005** (Phase 4 of the roadmap).
Designed for a free Colab/Kaggle T4 GPU (16 GB).

**Inputs** — supplied as a single tarball uploaded by you:

- `vn-news-dataset-v2.tar.gz` — produced by
  `python scripts/run_labeler.py --no-label --export v2` on the labeling box,
  then `tar -czvf` of `data/datasets/v2/` (1636 train / 218 val / 216 test rows,
  Gemini 2.5 Pro labels at prompt v1.2.0, 85.8 % QC pass rate).

**Outputs:**

- `models/vit5-news-v2/` — best LoRA checkpoint + tokenizer config.
- `mlruns/` — MLflow tracking dir with ROUGE / loss curves.
- `vit5-news-v2.tar.gz` — checkpoint tarball you can download via the Colab Files panel.

End-to-end on the v2 dataset (~1636 train rows, 4 epochs, batch=4, grad_accum=4, fp16): **~30–45 min** wall clock on a T4.

## 0. Verify the GPU is a T4 (or better)

In [ ]:
!nvidia-smi

## 1. Install pinned dependencies

These are the same versions used in the repo's `pyproject.toml` for the
training package, with one tweak: `bitsandbytes` is dropped because LoRA on
`vit5-base` doesn't need 8-bit quantisation to fit on a T4.

In [ ]:
!pip install -q \
    'transformers>=4.46,<5.0' \
    'datasets>=3.1' \
    'accelerate>=1.1' \
    'peft>=0.13' \
    'rouge-score>=0.1.2' \
    'sentencepiece>=0.2' \
    'mlflow>=2.17' \
    'PyYAML>=6.0' \
    'loguru>=0.7' \
    'bert-score>=0.3.13'

## 2. Clone the repo (read-only is fine — no git push needed)

In [ ]:
import os
REPO = 'https://github.com/khangnh22ds/vn-news-summarizer.git'
if not os.path.exists('vn-news-summarizer'):
    !git clone {REPO}
%cd vn-news-summarizer

## 3. Upload the dataset tarball

Run the next cell, then in the file picker that appears, select
`vn-news-dataset-v2.tar.gz` from your local disk. The file is ~2.6 MB so the
upload finishes in a few seconds.

Alternative: if you've already copied the tarball to Google Drive, comment
out the `files.upload()` block and uncomment the `drive.mount` block.

In [ ]:
from google.colab import files  # type: ignore
uploaded = files.upload()  # select vn-news-dataset-v2.tar.gz
assert 'vn-news-dataset-v2.tar.gz' in uploaded, list(uploaded)

# # Drive alternative — uncomment if you'd rather pull from Drive:
# from google.colab import drive  # type: ignore
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/vn-news/datasets/vn-news-dataset-v2.tar.gz .

In [ ]:
!mkdir -p data/datasets
!tar -xzf vn-news-dataset-v2.tar.gz -C data/datasets
!ls -lh data/datasets/v2/
!head -1 data/datasets/v2/train.jsonl | python -c "import sys, json; d=json.loads(sys.stdin.read()); print({k: (v[:120] + '...' if isinstance(v, str) and len(v) > 120 else v) for k, v in d.items()})"

## 4. Make the in-tree packages importable

The repo is a uv workspace; for a notebook we just point `sys.path` at
each `src/` directory directly so we can do `from vn_news_training import ...`
without running `uv sync`.

In [ ]:
import sys, pathlib
for p in [
    'packages/common/src',
    'packages/training/src',
    'packages/inference/src',
    'scripts',
]:
    sys.path.insert(0, str(pathlib.Path.cwd() / p))

from vn_news_training import load_finetune_config
cfg = load_finetune_config('configs/training/vit5_base_v2.yaml')
print('run_name      :', cfg.run_name)
print('dataset.name  :', cfg.dataset.name)
print('model.base    :', cfg.model.base)
print('output_dir    :', cfg.training.output_dir)
print('peft.enabled  :', cfg.peft.enabled, '(r=%d, alpha=%d)' % (cfg.peft.r, cfg.peft.alpha))

## 5. Run the fine-tune

`run_finetune` returns the `Trainer` object plus the final eval metrics on
the validation split. Internally it:

1. Loads the JSONL files into a `DatasetDict` and tokenises with the
   ViT5 SentencePiece tokenizer (`max_input_length=1024`,
   `max_target_length=128`).
2. Wraps `VietAI/vit5-base` with a LoRA adapter (`r=16`, `alpha=32`,
   targets the T5 attention `q` and `v` projections).
3. Trains for 4 epochs at fp16 with the Hugging Face `Seq2SeqTrainer`.
4. Computes ROUGE-1/2/L on the validation set every epoch and
   restores the best checkpoint at the end.

Expect a single epoch to take ~7–10 min on a T4. Total wall clock ~30–45 min.

In [ ]:
from vn_news_training.finetune import run_finetune
result = run_finetune('configs/training/vit5_base_v2.yaml')
result

## 6. Quick eval — fine-tuned model vs extractive baselines

Compare the fine-tuned LoRA checkpoint against `lexrank` and `textrank` on
the held-out test split. (We don't run BERTScore in this cell because
downloading the XLM-R model adds ~2 min and isn't necessary for the
ROUGE-vs-baseline comparison.)

In [ ]:
!python scripts/run_eval.py --baseline lexrank --dataset v2 --no-mlflow --split test
!python scripts/run_eval.py --baseline textrank --dataset v2 --no-mlflow --split test
!python scripts/run_eval.py --baseline vit5 --model-path models/vit5-news-v2 --dataset v2 --no-mlflow --split test

## 7. Download the checkpoint

Compress the LoRA adapter + tokenizer (~30 MB) and download it. You can
also push it back to Google Drive in the optional cell below.

In [ ]:
!tar -czvf vit5-news-v2.tar.gz models/vit5-news-v2
!ls -lh vit5-news-v2.tar.gz
from google.colab import files  # type: ignore
files.download('vit5-news-v2.tar.gz')

In [ ]:
# Optional — also save to Drive so a future Colab session can resume.
# from google.colab import drive  # type: ignore
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/vn-news/checkpoints/
# !cp vit5-news-v2.tar.gz /content/drive/MyDrive/vn-news/checkpoints/

## 8. (Optional) Print the final metrics summary

Pretty-print the ROUGE-1/2/L on the validation set + model size + epoch count
so you can paste them straight into the PR description.

In [ ]:
import json
summary = {
    'run_name': cfg.run_name,
    'output_dir': str(result.output_dir),
    'best_checkpoint': result.best_model_checkpoint,
    'metrics': result.metrics,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))